# 05 - Task 6: generation for DPO-vs-SFT comparisons
Attach outputs of notebooks 01, 02 and 03. Greedy decoding, repetition_penalty 1.1, max_new_tokens 512, identical for every model.

In [ ]:
# ---- Task 1: repository + environment setup (Kaggle, GPU T4 x2) ----
# Private repo: add a GitHub token as a Kaggle secret named GITHUB_TOKEN
# (Add-ons -> Secrets -> Add secret, then tick it for this notebook).
REPO_URL = "https://github.com/zainabusman22/advanced-genai-pa2.git"
REPO = "/kaggle/working/advanced-genai-pa2"

import os, subprocess, sys
if not os.path.exists(REPO):
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    authed = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
    subprocess.run(["git", "clone", "--depth", "1", authed, REPO], check=True, capture_output=True)
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", REPO_URL], check=True)  # don't keep the token on disk
    del token, authed
os.chdir(REPO)
!git -C {REPO} log --oneline -1
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest
!python -m pip install -q -e . --no-deps
!nvidia-smi

In [ ]:
import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() == 2, "Select the GPU T4 x2 accelerator"

In [ ]:
# Locate artifacts from earlier notebooks attached as input datasets.
import glob
def find_dir(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"nothing matching {pattern} under /kaggle/input - attach the previous notebook's output")
    print(pattern, "->", hits[0])
    return os.path.dirname(hits[0])

In [ ]:
SFT_BEST = find_dir("sft/best/config.json")
HH_BEST = find_dir("dpo_hh/best/config.json")
UF_BEST = find_dir("dpo_uf/best/config.json")
BASE_GENS = find_dir("generations/alpaca/base.json")  # baseline generations from notebook 01
!mkdir -p outputs/generations && cp -rn {BASE_GENS}/../* outputs/generations/ && ls -R outputs/generations

Sanity check (2-5 prompts) before the full run

In [ ]:
!python scripts/generate.py --model_path {UF_BEST} --eval_set eval_sets/alpaca_eval_300_seed42.json --generator_name s --output_path /tmp/s.json --limit 3
!python scripts/generate.py --model_path {HH_BEST} --eval_set eval_sets/hh_rlhf_eval_300_seed42.json --generator_name s --output_path /tmp/s.json --limit 3

Full generation: two GPUs in parallel

In [ ]:
import subprocess
G = "python scripts/generate.py"
gpu0 = (f"{G} --device cuda:0 --model_path {SFT_BEST} --eval_set eval_sets/alpaca_eval_300_seed42.json --generator_name qwen2.5-0.5b-sft --dataset_name alpaca_eval --output_path outputs/generations/alpaca/sft.json && "
        f"{G} --device cuda:0 --model_path {UF_BEST} --eval_set eval_sets/alpaca_eval_300_seed42.json --generator_name qwen2.5-0.5b-dpo-uf --dataset_name alpaca_eval --output_path outputs/generations/alpaca/dpo_uf.json")
gpu1 = (f"{G} --device cuda:1 --model_path {SFT_BEST} --eval_set eval_sets/hh_rlhf_eval_300_seed42.json --generator_name qwen2.5-0.5b-sft --output_path outputs/generations/hh/sft.json && "
        f"{G} --device cuda:1 --model_path {HH_BEST} --eval_set eval_sets/hh_rlhf_eval_300_seed42.json --generator_name qwen2.5-0.5b-dpo-hh --output_path outputs/generations/hh/dpo_hh.json")
procs = [subprocess.Popen(c, shell=True, stdout=open(f"/kaggle/working/gen{i}.log", "w"), stderr=subprocess.STDOUT) for i, c in enumerate([gpu0, gpu1])]
print([p.wait() for p in procs])
!tail -2 /kaggle/working/gen0.log /kaggle/working/gen1.log

Build blinded pairwise files (randomised A/B, hidden identity metadata)

In [ ]:
!python scripts/build_pairs.py --model_path outputs/generations/alpaca/dpo_uf.json --baseline_path outputs/generations/alpaca/sft.json --eval_kind general --output_path outputs/eval/uf_dpo_vs_sft_pairs.jsonl
!python scripts/build_pairs.py --model_path outputs/generations/hh/dpo_hh.json --baseline_path outputs/generations/hh/sft.json --eval_kind safety --output_path outputs/eval/hh_dpo_vs_sft_pairs.jsonl
!python scripts/build_pairs.py --model_path outputs/generations/alpaca/sft.json --baseline_path outputs/generations/alpaca/base.json --eval_kind general --output_path outputs/eval/sft_vs_base_alpaca_pairs.jsonl

In [ ]:
# Archive outputs so they survive the session (then: Save Version -> attach as dataset).
!cd {REPO} && zip -qr /kaggle/working/generation_outputs.zip outputs/generations outputs/eval
!ls -lh /kaggle/working/*.zip